In [2]:
import dask.array as da
import joblib
import xarray as xr
from dask_ml.wrappers import ParallelPostFit
from datacube.utils.geometry import assign_crs
from xarray import Dataset
from pystac_client import Client
from odc.stac import load
import os
import rasterio as rio
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from rasterio.transform import from_origin

from planetary_computer import sign_url

/tmp/ipykernel_976/2887921316.py:5: DeprecationWarning: datacube.utils.geometry is now deprecated. Please use the odc-geo library instead.
  from datacube.utils.geometry import assign_crs


In [3]:
DEP_CATALOG = "https://stac.digitalearthpacific.org"
# DEP_CATALOG = "https://stac.staging.digitalearthpacific.io"
MSPC_CATALOG = "https://planetarycomputer.microsoft.com/api/stac/v1/"

In [4]:
bbox = [178.42554, -17.69344, 178.54050, -17.59307]
chunks={"x": 2048, "y": 2048}

In [5]:
mspc_client = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1/")

# Get a pystac client for the MSPC
items_dem = list(mspc_client.search(collections=["nasadem"], bbox=bbox).items())


In [11]:
items_dem

[<Item id=NASADEM_HGT_s18e178>]

In [12]:
data = load(items_dem, measurements="elevation",
        bbox=bbox,
        chunks={"x": 2048, "y": 2048},
        groupby="solar_day",
    )

In [13]:
data_dem = load(items_dem, chunks=chunks, groupby="solar_day", like=data, patch_url=sign_url)

In [15]:
data_dem = (data_dem.where(data_dem != -32768).squeeze("time"))


In [16]:
data_dem

<xarray.Dataset> Size: 607kB
Dimensions:      (latitude: 362, longitude: 415)
Coordinates:
  * latitude     (latitude) float64 3kB -17.59 -17.59 -17.59 ... -17.69 -17.69
  * longitude    (longitude) float64 3kB 178.4 178.4 178.4 ... 178.5 178.5 178.5
    spatial_ref  int32 4B 4326
    time         datetime64[ns] 8B 2000-02-20
Data variables:
    elevation    (latitude, longitude) float32 601kB dask.array<chunksize=(362, 415), meta=np.ndarray>

In [25]:
# If you have a DataArray
data_dem['elevation'].plot(cmap='terrain')
plt.title("DEM Elevation")
plt.show()

In [18]:
dem = data_dem['elevation']

In [19]:
height, width = dem.shape
left, right, bottom, top = bbox

pixel_width = (right - left) / width
pixel_height = (top - bottom) / height

transform = from_origin(left, top, pixel_width, pixel_height)  # top left corner

with rio.open(
    "0-dem.tif", "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype=dem.dtype,
    crs="EPSG:32760",  # update if you know your DEM's CRS is different
    transform=transform,
    nodata=0  # or use -9999 if 0 is valid in your data
) as dst:
    dst.write(dem, 1)

In [26]:
ocean_mask = np.where(dem == 0, 1, 0)  # 1 for ocean, 0 for land

plt.imshow(ocean_mask, cmap='Blues')
plt.title('Ocean Mask')
plt.show()

In [27]:
nodata_value = -9999
dem_nodata = np.where(np.isnan(dem), nodata_value, dem).astype(np.float32)
plt.imshow(dem)

In [28]:
dem_masked = np.where(dem_nodata == 0, np.nan, dem_nodata)

# Example DEM array: ocean cells are np.nan, land is elevation
# dem_masked = np.where(dem_nodata == 0, np.nan, dem_nodata)

cmap = plt.get_cmap('terrain').copy()
cmap.set_bad(color='white')  # set color for ocean (masked np.nan)

im = plt.imshow(dem_masked, cmap=cmap)
plt.title('DEM with Ocean Masked')
cbar = plt.colorbar(im, label='Elevation (m)')